# SilentBridge — train real base_model.pt

One straight-line path: Install → Restart → Verify → Clone → Find dataset →
Prepare videos → Extract keypoints → Copy metadata → Validate → Train →
Validate model → Download.

Dataset: **ISL-CSLTR** (Kaggle `drblack00/isl-csltr-indian-sign-language-dataset`),
real sentence-level video clips (`Videos_Sentence_Level`, 687 mp4s). This
notebook only runs `DATA_MODE = "video"` in the main path — the word-frames
fallback (for a mirror that ships only isolated-word jpgs) is moved to the
appendix at the very end, out of the way.

Each STEP below is one or two cells. Run top to bottom. If a cell prints
`STOP —`, fix the named problem before continuing — don't run further cells.

**GPU**: this notebook cannot flip Colab's accelerator dropdown for you —
there's no API for that from inside a notebook. Do it once, manually, before
running anything: **Runtime > Change runtime type > T4 GPU > Save**. Cell 3
verifies it actually took.

## STEP 1 — Environment Setup

RUN THIS CELL

In [ ]:
# Clean install, one place. Tested-working combo for this extraction task:
# mediapipe==0.10.21 + protobuf==4.25.9 + numpy==1.26.4 — NOT installing
# tensorflow here; nothing in the extraction/training path needs it.
!pip uninstall -y mediapipe protobuf tensorflow tensorflow-cpu -q

!pip install -q --no-cache-dir \
    "numpy==1.26.4" \
    "protobuf==4.25.9" \
    "mediapipe==0.10.21" \
    "opencv-python-headless" \
    "pandas" \
    "kagglehub"

print("install done.")

## STEP 2 — Restart Required

**STOP — restart the runtime now**: Runtime > Restart session.

Do not run STEP 3 until you've restarted — the just-installed package
versions aren't loaded into this process until you do.

## STEP 3 — Verify Environment

RUN THIS CELL (after restarting)

In [ ]:
import numpy
import google.protobuf
import mediapipe as mp

print("NumPy:", numpy.__version__)
print("Protobuf:", google.protobuf.__version__)
print("MediaPipe:", mp.__version__)
print("Has solutions:", hasattr(mp, "solutions"))

assert hasattr(mp, "solutions"), (
    "STOP — mediapipe has no .solutions. The install in STEP 1 didn't take, "
    "or the runtime wasn't restarted. Re-run STEP 1, restart, re-run STEP 3."
)

with mp.solutions.holistic.Holistic(
    static_image_mode=False,
    model_complexity=1,
) as holistic:
    print("Holistic initialized OK")

import torch
gpu_ok = torch.cuda.is_available()
print("GPU available:", gpu_ok, "-", torch.cuda.get_device_name(0) if gpu_ok else "none")
if not gpu_ok:
    print("STOP (recommended) — no GPU. Runtime > Change runtime type > T4 GPU > Save, "
          "then Runtime > Restart session, then re-run from STEP 3. Training will still "
          "run on CPU if you skip this, just much slower.")

## STEP 4 — Clone / Locate SilentBridge

RUN THIS CELL

In [ ]:
import os

BASE_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO_ROOT = os.path.join(BASE_DIR, "SilentBridge")

if not os.path.isdir(REPO_ROOT):
    os.chdir(BASE_DIR)
    !git clone https://github.com/BharathWaj-K-R/SilentBridge.git
elif not os.path.isfile(os.path.join(REPO_ROOT, "README.md")):
    # exists but looks corrupted/incomplete — nuke and reclone
    !rm -rf {REPO_ROOT}
    os.chdir(BASE_DIR)
    !git clone https://github.com/BharathWaj-K-R/SilentBridge.git
else:
    # existing clean clone — update it. --ff-only refuses to silently run
    # stale code by failing loudly on any non-fast-forward state instead.
    !git -C {REPO_ROOT} pull --ff-only

os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)

script = open("backend/scripts/extract_keypoints.py", encoding="utf-8").read()
assert "_stub_tensorflow_for_mediapipe" in script, (
    "STOP — extract_keypoints.py is an older revision. git pull failed silently "
    "or REPO_ROOT points somewhere unexpected. Check the output above."
)
print("extract_keypoints.py revision OK")

## STEP 5 — Locate Dataset

RUN THIS CELL

In [ ]:
import glob

DATA_MODE = "video"

_kaggle_mounts = glob.glob("/kaggle/input/isl-csltr-indian-sign-language-dataset*")
if _kaggle_mounts:
    dataset_path = _kaggle_mounts[0]
    print("using existing Kaggle input mount (no download):", dataset_path)
else:
    import kagglehub
    dataset_path = kagglehub.dataset_download("drblack00/isl-csltr-indian-sign-language-dataset")
    print("downloaded to:", dataset_path)

# Find the sentence-level video directory — never guess silently. If this
# doesn't find exactly one candidate, it stops and prints what it did find
# so you can set VIDEO_ROOT yourself.
_candidates = [
    d for d in glob.glob(os.path.join(dataset_path, "**", "*Sentence_Level*"), recursive=True)
    if os.path.isdir(d) and "Video" in os.path.basename(d)
]

if len(_candidates) == 1:
    VIDEO_ROOT = _candidates[0]
    print("VIDEO_ROOT (auto-detected):", VIDEO_ROOT)
else:
    print(f"STOP — found {len(_candidates)} candidate video directories, expected exactly 1:")
    for c in _candidates:
        print(" ", c)
    print("Set VIDEO_ROOT manually in the next line, then re-run this cell body from here:")
    VIDEO_ROOT = None
    assert VIDEO_ROOT is not None, "Set VIDEO_ROOT above based on the candidates printed."

## STEP 6 — Prepare Videos + Labels

RUN THIS CELL

In [ ]:
import glob, os, shutil
import pandas as pd

os.chdir(REPO_ROOT)

RAW_VIDEOS_DIR = "data/raw_videos"
LABELS_CSV = "data/labels/ISLTranslate.csv"

shutil.rmtree(RAW_VIDEOS_DIR, ignore_errors=True)
os.makedirs(RAW_VIDEOS_DIR, exist_ok=True)
os.makedirs("data/labels", exist_ok=True)

video_files = []
for ext in ("mp4", "MP4", "avi", "AVI", "mov", "MOV"):
    video_files += glob.glob(os.path.join(VIDEO_ROOT, "**", f"*.{ext}"), recursive=True)

assert len(video_files) > 0, f"STOP — no video files found under VIDEO_ROOT: {VIDEO_ROOT}"
print(f"found {len(video_files)} video files")

# ADJUST if the sentence text isn't the immediate parent folder name
def sentence_id_from_path(video_path: str) -> str:
    return os.path.basename(os.path.dirname(video_path))

rows = []
for i, vp in enumerate(video_files):
    uid = f"clip{i:04d}"
    text = sentence_id_from_path(vp).replace("_", " ").strip()
    if not text:
        continue
    dst = os.path.join(RAW_VIDEOS_DIR, f"{uid}.mp4")
    shutil.copy(vp, dst)
    rows.append({"uid": uid, "text": text})

df = pd.DataFrame(rows)
df.to_csv(LABELS_CSV, index=False)

# --- validation, don't proceed silently on garbage ---
assert {"uid", "text"} <= set(df.columns), "STOP — CSV missing uid/text columns"
n_videos = len(glob.glob(os.path.join(RAW_VIDEOS_DIR, "*.mp4")))
assert len(df) == n_videos, (
    f"STOP — CSV has {len(df)} rows but {n_videos} videos were copied. Mismatch."
)
assert df["text"].str.len().min() > 0, "STOP — some rows have empty text."

print(f"{len(df)} rows written to {LABELS_CSV}, matches {n_videos} videos copied.")
print("\nsample rows:")
display(df.sample(min(5, len(df))))

## STEP 7 — Extract Keypoints

RUN THIS CELL (takes a while — MediaPipe runs per-frame on all clips)

In [ ]:
os.chdir(REPO_ROOT)
!python backend/scripts/extract_keypoints.py \
  --videos_dir data/raw_videos \
  --labels_csv data/labels/ISLTranslate.csv \
  --out_dir data/processed/isltranslate

## STEP 8 — Build Processed Metadata

RUN THIS CELL

In [ ]:
import shutil

os.chdir(REPO_ROOT)
shutil.copy("data/labels/ISLTranslate.csv", "data/processed/isltranslate/ISLTranslate.csv")
print("copied ISLTranslate.csv into data/processed/isltranslate/")

## STEP 9 — Validate Processed Dataset

RUN THIS CELL — training does not start unless this passes.

In [ ]:
import glob, sys

os.chdir(REPO_ROOT)

pose_files = glob.glob("data/processed/isltranslate/pose/*.npy")
face_files = glob.glob("data/processed/isltranslate/face/*.npy")
processed_csv = "data/processed/isltranslate/ISLTranslate.csv"

print("pose files:", len(pose_files))
print("face files:", len(face_files))
print("processed CSV exists:", os.path.isfile(processed_csv))

assert len(pose_files) > 0, "STOP — zero pose files. Check STEP 7's extraction output for errors."
assert len(face_files) > 0, "STOP — zero face files. Check STEP 7's extraction output for errors."
assert os.path.isfile(processed_csv), "STOP — processed CSV missing. Re-run STEP 8."

pose_uids = {os.path.splitext(os.path.basename(p))[0] for p in pose_files}
face_uids = {os.path.splitext(os.path.basename(p))[0] for p in face_files}
mismatched = pose_uids ^ face_uids
if mismatched:
    print(f"STOP — {len(mismatched)} UIDs have pose but not face (or vice versa): "
          f"{sorted(mismatched)[:10]}...")
    raise AssertionError("pose/face UID mismatch")

sys.path.insert(0, "backend")
from app.training.isltranslate import ISLTranslateKeypointDataset

dataset = ISLTranslateKeypointDataset("data/processed/isltranslate")
print(f"\nUsable examples: {len(dataset)}")
assert len(dataset) > 0, "STOP — 0 usable examples. See ISLTranslateKeypointDataset's filtering logic."

example = dataset[0]
print("\nsample example:")
print("  UID:       ", example["uid"])
print("  TEXT:      ", example["text"])
print("  POSE SHAPE:", tuple(example["pose"].shape))
print("  FACE SHAPE:", tuple(example["face"].shape))

## STEP 10 — Train

RUN THIS CELL

In [ ]:
%cd {REPO_ROOT}

!PYTHONPATH=backend python -m app.training.train_base_model \
  --data-dir data/processed/isltranslate \
  --output backend/app/models/weights/base_model.pt \
  --epochs 15 \
  --batch-size 4

## STEP 11 — Validate Outputs

RUN THIS CELL

In [ ]:
import os

os.chdir(REPO_ROOT)
PT_PATH = "backend/app/models/weights/base_model.pt"
VOCAB_PATH = "backend/app/models/weights/base_model.vocab.json"

assert os.path.isfile(PT_PATH), f"STOP — {PT_PATH} does not exist. Training did not complete."
assert os.path.isfile(VOCAB_PATH), f"STOP — {VOCAB_PATH} does not exist."

print(f"{PT_PATH}: {os.path.getsize(PT_PATH) / 1e6:.2f} MB")
print(f"{VOCAB_PATH}: {os.path.getsize(VOCAB_PATH) / 1e3:.2f} KB")

# actually load it with the repo's own code — file existing isn't success,
# loading successfully is.
import sys
sys.path.insert(0, "backend")
from app.models.base_model import load_frozen_base_model

try:
    model = load_frozen_base_model(PT_PATH)
    print("\nmodel loaded OK:", type(model).__name__)
except Exception as e:
    print(f"\nSTOP — file exists but failed to load: {e}")
    raise

## STEP 12 — Download Model

RUN THIS CELL

In [ ]:
PT_PATH = "backend/app/models/weights/base_model.pt"
VOCAB_PATH = "backend/app/models/weights/base_model.vocab.json"

try:
    from google.colab import files
    files.download(PT_PATH)
    files.download(VOCAB_PATH)
except ImportError:
    print("On Kaggle: download these two files from the notebook's Output pane —")
    print(PT_PATH)
    print(VOCAB_PATH)

print("drop both into backend/app/models/weights/ in your local repo, then commit+push.")

---
## Appendix — word_frames fallback (not part of the main path)

Only relevant if a future Kaggle mirror ships isolated-word jpgs instead of
real sentence videos (some mirrors of this dataset do — the one used above
does not). Not needed for the current run; do not execute this unless STEP 5
couldn't find `VIDEO_ROOT` and you've confirmed only word-level frames
exist.

In [ ]:
# Optional fallback — each word folder's jpgs become one short pseudo-clip.
# Set WORD_FRAMES_ROOT yourself if you actually need this path.
WORD_FRAMES_ROOT = None  # e.g. f"{dataset_path}/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/Frames_Word_Level"

if WORD_FRAMES_ROOT is None:
    print("skipped — set WORD_FRAMES_ROOT above to use this fallback")
else:
    import sys
    import numpy as np
    import cv2
    import mediapipe as mp

    if "tensorflow" not in sys.modules:
        try:
            import tensorflow  # noqa: F401
        except Exception:
            import types
            _fake_tf = types.ModuleType("tensorflow")
            _fake_tools = types.ModuleType("tensorflow.tools")
            _fake_docs = types.ModuleType("tensorflow.tools.docs")
            _fake_docs.doc_controls = types.SimpleNamespace(
                do_not_generate_docs=lambda f: f,
                for_subclass_implementers=lambda f: f,
                do_not_doc_inheritable=lambda f: f,
            )
            _fake_tf.tools = _fake_tools
            _fake_tools.docs = _fake_docs
            sys.modules["tensorflow"] = _fake_tf
            sys.modules["tensorflow.tools"] = _fake_tools
            sys.modules["tensorflow.tools.docs"] = _fake_docs

    out_dir = "data/processed/isltranslate"
    pose_dir = os.path.join(out_dir, "pose")
    face_dir = os.path.join(out_dir, "face")
    os.makedirs(pose_dir, exist_ok=True)
    os.makedirs(face_dir, exist_ok=True)

    mp_holistic = mp.solutions.holistic
    word_folders = sorted(
        d for d in os.listdir(WORD_FRAMES_ROOT)
        if os.path.isdir(os.path.join(WORD_FRAMES_ROOT, d))
    )
    print(f"found {len(word_folders)} word folders")

    rows = []
    with mp_holistic.Holistic(static_image_mode=True, model_complexity=1) as holistic:
        for word in word_folders:
            word_dir = os.path.join(WORD_FRAMES_ROOT, word)
            imgs = sorted(
                f for f in os.listdir(word_dir)
                if f.lower().endswith((".jpg", ".jpeg", ".png"))
            )
            if not imgs:
                continue
            pose_frames, face_frames = [], []
            for img_name in imgs:
                img = cv2.imread(os.path.join(word_dir, img_name))
                if img is None:
                    continue
                rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                results = holistic.process(rgb)
                if results.pose_landmarks:
                    pose_frames.append(np.array(
                        [[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark],
                        dtype=np.float32).flatten())
                else:
                    pose_frames.append(np.zeros(33 * 4, dtype=np.float32))
                if results.face_landmarks:
                    face_frames.append(np.array(
                        [[lm.x, lm.y, lm.z] for lm in results.face_landmarks.landmark],
                        dtype=np.float32).flatten())
                else:
                    face_frames.append(np.zeros(478 * 3, dtype=np.float32))
            if not pose_frames:
                continue
            uid = word.strip().lower().replace(" ", "_").replace("/", "_").replace("'", "")
            np.save(os.path.join(pose_dir, f"{uid}.npy"), np.stack(pose_frames))
            np.save(os.path.join(face_dir, f"{uid}.npy"), np.stack(face_frames))
            rows.append({"uid": uid, "text": word.strip().lower().replace("_", " ")})
            print(f"[{len(rows)}] {word} -> {len(pose_frames)} frames")

    import pandas as pd
    pd.DataFrame(rows).to_csv(os.path.join(out_dir, "ISLTranslate.csv"), index=False)
    print("done — now go back to STEP 9 to validate.")

---
## Utility — reset, start clean

Optional. Wipes everything this notebook wrote (raw videos, labels,
processed features, trained weights) but leaves the read-only dataset mount
untouched, so STEP 5 won't need to re-download anything after this.

In [ ]:
import shutil

os.chdir(REPO_ROOT)
for path in [
    "data/raw_videos",
    "data/labels",
    "data/processed",
    "backend/app/models/weights/base_model.pt",
    "backend/app/models/weights/base_model.vocab.json",
]:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print("removed dir:", path)
    elif os.path.isfile(path):
        os.remove(path)
        print("removed file:", path)
print("clean. re-run from STEP 5 onward.")